# SANATIO AI Server
Run all cells top to bottom. Keep open while using your website.

In [ ]:
!pip install torch torchvision opencv-python-headless numpy pillow pyngrok -q
print('Packages installed')

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Model uploaded' if 'ai_detector_model.pth' in uploaded else 'Upload ai_detector_model.pth')

In [ ]:
server_code = '''
import os, base64, json
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse
import cv2, numpy as np, torch, torch.nn as nn
from torchvision import models

MODEL_PATH = "ai_detector_model.pth"
IMAGE_SIZE = 224
PORT = 5050
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading model on {DEVICE}...")
model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Model ready!")

def predict(image_bytes):
    arr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None: raise ValueError("Could not decode image")
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    real_score = round(probs[0].item() * 100, 1)
    ai_score = round(probs[1].item() * 100, 1)
    return {"aiScore": ai_score, "realScore": real_score, "likelyLabel": "Likely AI-generated" if ai_score >= 50 else "Likely real"}

class Handler(BaseHTTPRequestHandler):
    def log_message(self, format, *args): pass
    def _cors(self):
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "POST, GET, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, ngrok-skip-browser-warning")
        self.send_header("ngrok-skip-browser-warning", "69420")
    def do_OPTIONS(self):
        self.send_response(200)
        self._cors()
        self.end_headers()
    def do_GET(self):
        body = json.dumps({"ok": True}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self._cors()
        self.end_headers()
        self.wfile.write(body)
    def do_POST(self):
        path = urlparse(self.path).path
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length))
        if path == "/analyze":
            try:
                data_url = body.get("image", "")
                if "," in data_url: data_url = data_url.split(",", 1)[1]
                result = predict(base64.b64decode(data_url))
                self._json(200, result)
                print(f"  {result[chr(39)]likelyLabel{chr(39)}} AI:{result[chr(39)]aiScore{chr(39)]}%")
            except Exception as e:
                self._json(500, {"error": str(e)})
        else:
            self.send_response(404); self.end_headers()
    def _json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self._cors()
        self.end_headers()
        self.wfile.write(body)

httpd = HTTPServer(("0.0.0.0", PORT), Handler)
print(f"Server running on port {PORT}")
httpd.serve_forever()
'''
with open('server.py', 'w') as f:
    f.write(server_code)
print('server.py written')

In [ ]:
# Paste your ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"

from pyngrok import ngrok
import threading, time

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

def run_server():
    exec(open('server.py').read(), {})

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(6)

tunnel = ngrok.connect(5050, bind_tls=True)
public_url = tunnel.public_url

print('='*55)
print('SANATIO AI Server is LIVE!')
print('='*55)
print(f'Your public URL: {public_url}')
print(f'Copy into detector-server.js:')
print(f'const SERVER_URL = "{public_url}/analyze";')
print('Keep this tab open while using your website!')